# UMI in MuJoCo - Phase 0: Recon (Ubuntu / local GPU version)

Run this in Jupyter (JupyterLab or VS Code) on your Ubuntu machine, inside the Python env you plan to use for the project (conda or venv).

Goal: check hardware, clone the UMI repo, read the key configs, check dataset size, and inspect the zarr format your MuJoCo demos must match later.

## 0. Settings (edit these)

In [ ]:
import os
from pathlib import Path

# Where everything lives. Change if you want it elsewhere.
WORKDIR = Path.home() / 'umi_work'
WORKDIR.mkdir(parents=True, exist_ok=True)
REPO = WORKDIR / 'universal_manipulation_interface'
DATASET_URL = 'https://real.stanford.edu/umi/data/zarr_datasets/cup_in_the_wild.zarr.zip'
DATASET = WORKDIR / 'cup_in_the_wild.zarr.zip'
print('WORKDIR:', WORKDIR)

## 1. Check GPU, CUDA and Python

In [ ]:
!nvidia-smi
import sys, platform
print(sys.version)
print(platform.platform())
!nvcc --version 2>/dev/null || echo 'nvcc not found (fine if you use pip/conda PyTorch wheels)'
try:
    import torch
    print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print('GPU: %s, %.1f GB VRAM' % (p.name, p.total_memory / 1e9))
except ImportError:
    print('torch not installed yet (Phase 1 handles this)')

## 2. Clone the repo (safe to re-run)

In [ ]:
if not REPO.exists():
    !git clone https://github.com/real-stanford/universal_manipulation_interface.git {REPO}
else:
    print('Repo already exists, skipping clone')
os.chdir(REPO)
print('cwd:', os.getcwd())
!git log -1 --format='%h %ad %s'

## 3. Map the repo (what to keep vs skip)

In [ ]:
!ls
!echo '--- configs ---'
!find . -name '*.yaml' | grep -i -E 'umi|train_diffusion_unet_timm' | head -30
!echo '--- SLAM pipeline scripts (SKIP in sim) ---'
!ls scripts_slam_pipeline
!echo '--- umi package ---'
!ls umi umi/real_world

## 4. Read the training and task configs
Note `shape_meta` (observation keys, shapes, horizons). Your sim data must match this.

In [ ]:
import glob
for pattern in ['**/train_diffusion_unet_timm_umi_workspace.yaml', '**/umi.yaml']:
    for p in glob.glob(pattern, recursive=True):
        print('=' * 20, p, '=' * 20)
        print(''.join(open(p).readlines()[:150]))

## 5. Find the exact data keys the replay-buffer script writes

In [ ]:
!grep -n -E 'robot0|gripper|episode_ends|camera0_rgb|eef' scripts_slam_pipeline/07_generate_replay_buffer.py | head -50

## 6. Check dataset size vs free disk

In [ ]:
import subprocess, shutil
out = subprocess.run(['curl', '-sIL', DATASET_URL], capture_output=True, text=True).stdout
sizes = [int(l.split(':')[1]) for l in out.splitlines() if l.lower().startswith('content-length')]
size_gb = sizes[-1] / 1e9 if sizes else None
free_gb = shutil.disk_usage(WORKDIR).free / 1e9
print('Dataset size: %s GB' % (round(size_gb, 2) if size_gb else 'unknown'))
print('Free disk at WORKDIR: %.1f GB' % free_gb)
!df -h {WORKDIR}

## 7. Download the dataset (resumable)
Auto-enables only if the size is known and fits with 2x headroom (you'll want room for extraction/caches later).

In [ ]:
DOWNLOAD = bool(size_gb) and free_gb > 2 * size_gb   # set True/False manually to override
print('DOWNLOAD =', DOWNLOAD)
if DOWNLOAD:
    !wget -c -O {DATASET} {DATASET_URL}
    !ls -lh {DATASET}

## 8. Inspect the zarr structure
This is the format your MuJoCo demos must reproduce.

In [ ]:
%pip -q install 'zarr<3' numcodecs imagecodecs
import zarr
store = zarr.ZipStore(str(DATASET), mode='r')
root = zarr.group(store=store)

def show(g, indent=0):
    for k, v in g.items():
        if hasattr(v, 'shape'):
            print(' ' * indent + '%s  shape=%s dtype=%s chunks=%s' % (k, v.shape, v.dtype, v.chunks))
        else:
            print(' ' * indent + k + '/')
            show(v, indent + 2)

show(root)

## 9. Episode statistics

In [ ]:
import numpy as np
ends = root['meta/episode_ends'][:]
lengths = np.diff(np.concatenate([[0], ends]))
print('episodes:', len(ends))
print('total frames:', int(ends[-1]))
print('episode length min/mean/max:', lengths.min(), round(lengths.mean(), 1), lengths.max())

## 10. Look at a few frames
Compare these real fisheye images with what your MuJoCo wrist camera will render (the domain gap).

In [ ]:
!grep -rn 'def register_codecs' --include=*.py .

In [ ]:
import sys
sys.path.insert(0, os.getcwd())
try:
    from diffusion_policy.codecs.imagecodecs_numcodecs import register_codecs
except ImportError:
    from umi.codecs.imagecodecs_numcodecs import register_codecs
register_codecs()

import matplotlib.pyplot as plt
rgb_keys = [k for k in root['data'].keys() if 'rgb' in k]
print('image keys:', rgb_keys)
imgs = root['data'][rgb_keys[0]]
idx = np.linspace(0, int(ends[0]) - 1, 4).astype(int)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, i in zip(axes, idx):
    ax.imshow(imgs[i]); ax.set_title('frame %d' % i); ax.axis('off')
plt.show()

## Phase 0 checklist (fill in)

- GPU model and memory: ____
- Dataset size vs free disk: ____
- Observation keys and shapes (from step 8): ____
- Image resolution and episode length (from steps 8-9): ____
- Keep: Diffusion Policy training code, zarr format, relative-pose actions, eval loop structure
- Skip: GoPro, SLAM, IMU sync, UR5/Franka drivers, WSG gripper, SpaceMouse

Next: Phase 1 (environment install), then Phase 2 (MuJoCo scene).